# Final comparison

Master table, bootstrap confidence intervals, exact McNemar tests, charts and a sample of disagreements for manual review. CPU runtime.

In [ ]:
# ===== STEP 7: final comparison - master table, significance tests, charts, error review =====
# CPU runtime is fine. Re-run after the Gemini step finishes and it will be included automatically.
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import f1_score, accuracy_score

DATA = "/content/drive/MyDrive/sci-classify/data"
RES = "/content/drive/MyDrive/sci-classify/results"
from google.colab import drive
drive.mount("/content/drive")
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")
gold = test_llm.label.tolist()
load = lambda f: json.load(open(f"{RES}/{f}")) if os.path.exists(f"{RES}/{f}") else None

# ---------- collect every model's predictions on the same 1,000 sentences ----------
models = {}    # name -> dict(preds, labels_used, ms, full_f1)
s2 = load("step2_preds_1000.json")
if s2:
    models["TF-IDF (text)"] = {"preds": s2["preds"]["TF-IDF + logistic regression"], "labels_used": 180040}
    models["TF-IDF + position"] = {"preds": s2["preds"]["TF-IDF + position + logistic regression"], "labels_used": 180040, "full_f1": None}
s3 = load("step3_setfit.json") or []
curve = pd.DataFrame([{"n": r["n_per_class"], "pos": "position" in r["model"], "f1": r["macro_f1_1000"]} for r in s3])
for r in s3:
    if r["n_per_class"] == 512:
        key = "SetFit 512/class" + (" + position" if "position" in r["model"] else "")
        models[key] = {"preds": r["preds_1000"], "labels_used": 2560, "ms": r.get("ms_per_sentence"),
                       "full_f1": r.get("macro_f1_full")}
s4 = load("step4_finetune.json")
if s4:
    models["BiomedBERT fine-tuned + position"] = {"preds": s4["preds_1000"], "labels_used": 180040,
                                                  "ms": s4["ms_per_sentence"], "full_f1": s4["macro_f1_full"]}
s4b = load("step4b_finetune_weighted.json")
if s4b:
    models["BiomedBERT fine-tuned + position + class weights"] = {"preds": s4b["preds_1000"], "labels_used": 180040,
                                                                  "ms": s4b["ms_per_sentence"], "full_f1": s4b["macro_f1_full"]}
s6 = load("step6_open_llm_all.json")
OPEN_LLM_MS = {"open_llm_zero_shot": 302, "open_llm_zero_shot_position": 299, "open_llm_few_shot_position": 380}  # from Step 6 output
if s6:
    short = s6["model"].split("/")[-1]
    for k, v in s6["preds"].items():
        models[f"{short}: {k.replace('open_llm_', '').replace('_', ' ')}"] = {"preds": v, "labels_used": 10 if "few" in k else 0,
                                                                            "ms": OPEN_LLM_MS.get(k)}
s6b = load("step6b_open_llm_14b.json")
if s6b:
    short = s6b["model"].split("/")[-1]
    for k, v in s6b["preds"].items():
        models[f"{short}: {k.replace('open_llm_', '').replace('_', ' ')}"] = {"preds": v, "labels_used": 10 if "few" in k else 0,
                                                                            "ms": s6b.get("ms", {}).get(k)}
s5 = load("step5_gemini_all.json")
if s5:
    for k, v in s5["preds"].items():
        models[f"{s5['model']}: {k.replace('gemini_', '').replace('_', ' ')}"] = {"preds": v, "labels_used": 10 if "few" in k else 0}
else:
    print("Gemini results not found yet - they will be added when you re-run this after Step 5.\n")

# ---------- master table with bootstrap confidence intervals ----------
def boot(p, n=1000, seed=0):
    rng = np.random.default_rng(seed); y, q = np.asarray(gold), np.asarray(p); idx = np.arange(len(y))
    s = [f1_score(y[i], q[i], average="macro", labels=LABELS) for i in (rng.choice(idx, len(idx)) for _ in range(n))]
    return np.percentile(s, [2.5, 97.5])

rows = []
for name, m in models.items():
    lo, hi = boot(m["preds"])
    rows.append({"model": name, "labelled examples": m["labels_used"],
                 "macro-F1 (1000 set)": f1_score(gold, m["preds"], average="macro", labels=LABELS),
                 "95% CI": f"{lo:.3f} to {hi:.3f}", "accuracy": accuracy_score(gold, m["preds"]),
                 "OBJECTIVE F1": f1_score(gold, m["preds"], labels=["OBJECTIVE"], average="macro"),
                 "invalid": sum(p not in LABELS for p in m["preds"]),
                 "ms/sentence": m.get("ms"), "full-test macro-F1": m.get("full_f1")})
table = pd.DataFrame(rows).sort_values("macro-F1 (1000 set)", ascending=False)
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 48)
print("=== Master results (same 1,000 balanced test sentences) ===")
print(table.round(3).to_string(index=False))
table.to_csv(f"{RES}/final_results_table.csv", index=False)

# ---------- paired significance: every model against the best one (exact McNemar test) ----------
best = table.iloc[0].model
print(f"\n=== Is the best model ({best}) significantly better? Exact McNemar test, paired on the same sentences ===")
bp = np.array(models[best]["preds"]) == np.array(gold)
for name in table.model[1:]:
    op = np.array(models[name]["preds"]) == np.array(gold)
    b, c = int(np.sum(bp & ~op)), int(np.sum(~bp & op))
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    print(f"  vs {name:45s} best right/other wrong {b:4d} | other right/best wrong {c:4d} | p = {p:.2g}")

# ---------- chart 1: how much labelled data do you need? ----------
fig, ax = plt.subplots(figsize=(9, 5))
txt = curve[~curve.pos].groupby("n").f1.agg(["mean", "std"]).reset_index()
ax.errorbar(txt.n * 5, txt["mean"], yerr=txt["std"].fillna(0), marker="o", capsize=4, label="SetFit (text only)")
for name, colour, ls in [("TF-IDF (text)", "grey", ":"), ("TF-IDF + position", "black", ":"),
                         ("BiomedBERT fine-tuned + position", "#12A08A", "--")]:
    if name in models:
        ax.axhline(table.set_index("model").loc[name, "macro-F1 (1000 set)"], color=colour, ls=ls,
                   label=f"{name} (180,040 labels)")
llms = table[table["labelled examples"] <= 10]
if len(llms):
    top = llms.iloc[0]
    ax.axhline(top["macro-F1 (1000 set)"], color="#E5484D", ls="-.", label=f"Best LLM: {top.model}")
ax.set_xscale("log"); ax.set_xlabel("Labelled training sentences (log scale)"); ax.set_ylabel("Macro-F1 (1,000-sentence test set)")
ax.set_title("How much labelled data does a small model need?"); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(f"{RES}/chart_learning_curve.png", dpi=150)

# ---------- chart 2: accuracy against speed ----------
sp = table.dropna(subset=["ms/sentence"])
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sp["ms/sentence"], sp["macro-F1 (1000 set)"], s=80, color="#1565C0")
for r in sp.itertuples():
    ax.annotate(r.model, (r._8, r._3), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log"); ax.set_xlim(right=sp["ms/sentence"].max() * 12)
ax.set_xlabel("Milliseconds per sentence on a free T4 GPU (log scale)")
ax.set_ylabel("Macro-F1"); ax.set_title("Accuracy against speed"); ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(f"{RES}/chart_accuracy_vs_speed.png", dpi=150)

# ---------- error review: are the "errors" really errors? ----------
bpred = models[best]["preds"]
wrong = test_llm.assign(model_prediction=bpred)[np.array(bpred) != np.array(gold)]
sample = wrong.sample(min(30, len(wrong)), random_state=1)[["abstract_id", "line_number", "total_lines", "text", "label", "model_prediction"]]
sample = sample.rename(columns={"label": "gold_label"}); sample["your_judgement"] = ""
out_csv = f"{RES}/error_review_30.csv"
if os.path.exists(out_csv):                      # never overwrite a file you may have already judged
    out_csv = f"{RES}/error_review_30_new_{best.split(' ')[0]}_{len(models)}models.csv"
sample.to_csv(out_csv, index=False)
print(f"\n=== {len(wrong)} disagreements for the best model; 30 saved to {os.path.basename(out_csv)} ===")
for r in sample.head(8).itertuples():
    print(f"  gold {r.gold_label:11s} model {r.model_prediction:11s} | {r.text[:95]}")
print("\nSaved: final_results_table.csv, chart_learning_curve.png, chart_accuracy_vs_speed.png, error_review_30.csv")
